<a href="https://colab.research.google.com/github/AlanChi0720/bio_ai/blob/main/B3_zero_shot_mutations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Track B — Notebook 3: Zero-shot Mutation Effect Prediction

**Big idea:** ESM-2 was trained as a masked language model on 250M protein sequences. At every position, given the surrounding context, it learned to predict which amino acid should be there. We can use that probability *directly* as a fitness predictor — **no training, no labels**.

Specifically, the score for a mutation `A → B` at position `i` is:

$$\text{score}(A_i \to B_i) = \log P(B_i \mid \text{context}) - \log P(A_i \mid \text{context})$$

If the model thinks the mutant is more likely than the wild-type, the mutation is probably tolerated. If much less likely, it probably breaks the protein. This works because evolutionary pressure (the source of the training data) selects functional sequences.

**What we'll do:**
1. Pick a wild-type protein (β-lactamase, the classic enzyme in DMS studies)
2. Score every possible single-residue substitution
3. Visualize the 20 × L mutation landscape
4. Compare against a real Deep Mutational Scanning (DMS) experiment

**Estimated time:** ~2-3 hours, GPU recommended.

In [ ]:
!pip install -q transformers torch

In [ ]:
import io, urllib.request, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import EsmTokenizer, EsmForMaskedLM
from scipy.stats import spearmanr, pearsonr

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
sns.set_theme(style='whitegrid')

## 1. Wild-type Sequence

**TEM-1 β-lactamase** (UniProt P62593, BLAT_ECOLX). This bacterial enzyme inactivates penicillin-class antibiotics by hydrolyzing the β-lactam ring. It's the most-studied protein in deep mutational scanning.

In [ ]:
# TEM-1 β-lactamase mature sequence (residues 24-286, after signal peptide)
WT_SEQ = (
    'HPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRIDAGQEQLGRRIH'
    'YSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEP'
    'ELNEAIPNDERDTTMPVAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWFIADKSGAG'
    'ERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW'
)
L = len(WT_SEQ)
print(f'WT length: {L} residues')

## 2. Load ESM-2 with Masked-LM Head

Note we use `EsmForMaskedLM` (not `EsmModel`) — this gives us access to the prediction logits over the 33-token alphabet for every position.

In [ ]:
MODEL_NAME = 'facebook/esm2_t6_8M_UR50D'   # 8M params — fast. Try t12 (35M) if you have time.
tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)
mlm = EsmForMaskedLM.from_pretrained(MODEL_NAME).to(device)
mlm.eval()
for p in mlm.parameters(): p.requires_grad = False

AAs = list('ACDEFGHIKLMNPQRSTVWY')   # 20 standard amino acids
aa_token_ids = torch.tensor(tokenizer.convert_tokens_to_ids(AAs), device=device)
print('AA token ids:', aa_token_ids.cpu().numpy())

## 3. Compute the Mutation Score Matrix

For each position, replace the wild-type residue with `<mask>`, run ESM-2, and read off the log-probabilities of all 20 amino acids.

Mathematically:
- Let $p_i(B)$ = log-probability that ESM-2 assigns to residue B at position i (given the rest of the sequence)
- Score for mutation $A \to B$ at position i: $p_i(B) - p_i(A)$

In [ ]:
@torch.no_grad()
def mutation_scores(wt_seq, mlm, tokenizer, batch_size=8):
    """Returns (L, 20) matrix where entry [i, k] = score of WT[i] -> AAs[k]."""
    mask_id = tokenizer.mask_token_id
    L = len(wt_seq)
    scores = np.zeros((L, 20), dtype=np.float32)
    wt_ids = tokenizer(wt_seq, add_special_tokens=True).input_ids  # list of token ids

    # Build L masked variants and run them through the model in batches
    masked_seqs = []
    for i in range(L):
        ids = wt_ids.copy()
        ids[i + 1] = mask_id              # +1 because of the leading <cls> token
        masked_seqs.append(ids)

    for start in range(0, L, batch_size):
        batch = masked_seqs[start:start + batch_size]
        ids = torch.tensor(batch, device=device)
        attn = torch.ones_like(ids)
        logits = mlm(input_ids=ids, attention_mask=attn).logits   # (B, L+2, vocab)
        log_probs = F.log_softmax(logits, dim=-1)
        for j, i in enumerate(range(start, min(start + batch_size, L))):
            wt_aa_id = wt_ids[i + 1]
            wt_logp = log_probs[j, i + 1, wt_aa_id]
            mut_logp = log_probs[j, i + 1, aa_token_ids]   # (20,)
            scores[i] = (mut_logp - wt_logp).cpu().numpy()
    return scores

t0 = time.time()
S = mutation_scores(WT_SEQ, mlm, tokenizer)
print(f'Computed {S.size} mutation scores in {time.time()-t0:.1f}s')
print(f'Score matrix shape: {S.shape}  (positions, amino acids)')

## 4. Visualize the Mutation Landscape

Plot the L × 20 matrix as a heatmap. Blue = mutation predicted to be tolerated; red = predicted to be deleterious. Black squares mark the wild-type.

In [ ]:
# Show first 100 residues
WINDOW = (0,263)
df_S = pd.DataFrame(S[WINDOW[0]:WINDOW[1]].T, index=AAs)

fig, ax = plt.subplots(figsize=(18, 5))
vmax = max(abs(S.min()), abs(S.max()))
sns.heatmap(df_S, cmap='RdBu', center=0, vmin=-vmax, vmax=vmax,
            cbar_kws={'label': 'log P(mut) - log P(wt)'}, ax=ax)

# Mark the wild-type with a black dot at each position
for i, pos in enumerate(range(WINDOW[0], WINDOW[1])):
    wt_aa = WT_SEQ[pos]
    if wt_aa in AAs:
        ax.plot(i + 0.5, AAs.index(wt_aa) + 0.5, 'k.', markersize=4)

ax.set_xlabel('Position'); ax.set_ylabel('Mutant amino acid')
ax.set_title(f'ESM-2 zero-shot mutation effects (residues {WINDOW[0]+1}-{WINDOW[1]})')
plt.tight_layout(); plt.show()

## 5. Per-Position Tolerance

How tolerant is each position? Sum over the 19 non-WT mutations. Highly conserved positions show very negative tolerance — these are the residues evolution rarely changes, often catalytic or structural anchors.

In [ ]:
# Mean score over the 19 non-WT amino acids at each position
wt_idx = np.array([AAs.index(a) if a in AAs else -1 for a in WT_SEQ])
mask = np.ones_like(S, dtype=bool)
for i, k in enumerate(wt_idx):
    if k >= 0: mask[i, k] = False
tol = np.nanmean(np.where(mask, S, np.nan), axis=1)   # mean log-prob change

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(range(1, L + 1), tol, lw=1)
ax.axhline(0, color='k', ls='--', lw=0.7)
ax.fill_between(range(1, L + 1), tol, 0, where=(tol < 0), color='red', alpha=0.3, label='conserved')
ax.fill_between(range(1, L + 1), tol, 0, where=(tol > 0), color='blue', alpha=0.3, label='tolerant')
ax.set_xlabel('Position'); ax.set_ylabel('Mean mutation score')
ax.set_title('Per-position evolutionary tolerance (negative = conserved)')
ax.legend(); plt.tight_layout(); plt.show()

# Q: For TEM-1, the catalytic Ser70 (index 69 in 0-indexed; ~70 in plot) should be highly conserved.
# Check the dip — does ESM-2 spot it?
print(f'Score at position 70 (catalytic Ser): {tol[69]:.2f}')
print(f'Mean position score: {np.nanmean(tol):.2f}')

In [ ]:
print(f'Ser70 best possible substitution: {S[69].max():.2f}')
print(f'Ser70 worst possible substitution: {S[69].min():.2f}')

In [ ]:
# 最保守的前 10 個位置（tol 最負）
order = np.argsort(tol)          # 由小到大排序後的 index
for rank, idx in enumerate(order[:10], 1):
    print(f'{rank:2d}. position {idx+1:3d}  ({WT_SEQ[idx]})  tol = {tol[idx]:.2f}')

In [ ]:
# 最「容忍/有利」的前 10 個位置（tol 最正）
for rank, idx in enumerate(order[::-1][:10], 1):
    print(f'{rank:2d}. position {idx+1:3d}  ({WT_SEQ[idx]})  tol = {tol[idx]:.2f}')

## 6. Top Predictions — What Does ESM-2 Think Is Beneficial vs Harmful?

In [ ]:
# Build a flat table of all (L * 20) mutations
rows = []
for i in range(L):
    for k, aa in enumerate(AAs):
        if WT_SEQ[i] == aa: continue   # skip WT
        rows.append({
            'pos': i + 1, 'wt': WT_SEQ[i], 'mut': aa,
            'mutation': f'{WT_SEQ[i]}{i+1}{aa}',
            'score': S[i, k],
        })
muts = pd.DataFrame(rows)

print('Top 10 PREDICTED-BENEFICIAL mutations:')
print(muts.nlargest(10, 'score')[['mutation', 'score']].to_string(index=False))
print('\nTop 10 PREDICTED-DELETERIOUS mutations:')
print(muts.nsmallest(10, 'score')[['mutation', 'score']].to_string(index=False))

## 7. Validate Against a Real DMS Experiment

Stiffler et al. 2015 measured the ampicillin resistance of nearly every single point mutation in TEM-1. We compare ESM-2's zero-shot scores to those experimental fitness values.

If you can't download the DMS data (URL or access changes), this section will skip but the rest of the notebook still runs.

In [ ]:
# Try to fetch the BLAT_ECOLX_Stiffler_2015 DMS data from ProteinGym
# If this URL no longer works, you can manually download from https://proteingym.org/
DMS_URL = 'https://huggingface.co/datasets/OATML-Markslab/ProteinGym_v0.1/resolve/main/ProteinGym_substitutions/BLAT_ECOLX_Stiffler_2015.csv'

dms_df = None
try:
    dms_df = pd.read_csv(DMS_URL)
    print(f'Loaded {len(dms_df)} DMS measurements.')
    print('Columns:', dms_df.columns.tolist())
    dms_df.head()
except Exception as e:
    print('DMS download failed:', e)
    print('Skip section 7 or upload a DMS CSV manually with columns mutant, DMS_score.')

In [ ]:
dms_df.head()

In [ ]:
if dms_df is not None:
    score_col = 'DMS_score' if 'DMS_score' in dms_df.columns else dms_df.columns[1]
    mut_col = 'mutant' if 'mutant' in dms_df.columns else dms_df.columns[0]

    def parse_mutation(m):
        wt_aa, pos, mut_aa = m[0], int(m[1:-1]), m[-1]
        return wt_aa, pos, mut_aa

    # ---- 新增：先自動找出 offset ----
    dms_wt = {}
    for m in dms_df[mut_col]:
        try:
            dms_wt[int(m[1:-1])] = m[0]
        except Exception:
            pass
    positions = sorted(dms_wt)
    best_off, best_hits = 0, -1
    for off in range(-30, 31):
        hits = sum(1 for p in positions
                   if 0 <= p - 1 + off < L and WT_SEQ[p - 1 + off] == dms_wt[p])
        if hits > best_hits:
            best_off, best_hits = off, hits
    print(f'Best offset = {best_off}  ({best_hits}/{len(positions)} positions match)')
    # --------------------------------

    dms_df['esm2_score'] = np.nan
    matched = 0
    for idx, row in dms_df.iterrows():
        try:
            wt_aa, pos, mut_aa = parse_mutation(row[mut_col])
            j = pos - 1 + best_off                          # ← 改這行（原本是 pos - 1）
            if 0 <= j < L and WT_SEQ[j] == wt_aa and mut_aa in AAs:
                dms_df.at[idx, 'esm2_score'] = S[j, AAs.index(mut_aa)]   # ← 這行的 pos-1 也換成 j
                matched += 1
        except Exception:
            pass

    paired = dms_df.dropna(subset=['esm2_score', score_col])
    rho = spearmanr(paired[score_col], paired['esm2_score']).correlation
    r = pearsonr(paired[score_col], paired['esm2_score'])[0]
    print(f'Matched mutations: {matched} / {len(dms_df)}')
    print(f'Spearman correlation: {rho:.3f}')
    print(f'Pearson correlation:  {r:.3f}')

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(paired[score_col], paired['esm2_score'], alpha=0.3, s=8)
    ax.set_xlabel('Experimental DMS score (fitness)')
    ax.set_ylabel('ESM-2 zero-shot score')
    ax.set_title(f'ESM-2 zero-shot vs experiment\nSpearman={rho:.3f},  Pearson={r:.3f}')
    plt.tight_layout(); plt.show()
else:
    print('Skipping validation — no DMS data loaded.')

## 8. Try Your Own Protein

Replace `WT_SEQ` above with any protein you care about (no longer than ~500 AA for the small ESM-2 model). Some ideas:
- A drug target (e.g., a kinase you study)
- A disease-associated protein (BRCA1, p53, CFTR)
- Your own designed sequence

The mutation heatmap will tell you which residues are evolutionarily constrained.

## Reflection Questions

1. **Why does this work without training?** ESM-2 was never told what "fitness" is. So what does it actually predict, and why does that correlate with experimental fitness?
2. **Where it fails** — Zero-shot ESM-2 scores correlate with fitness around 0.4-0.6 Spearman across most DMS datasets. What kinds of mutations would it most likely get wrong? (Hint: think about epistasis, allosteric sites, gain-of-function.)
3. **Comparison to B1/B2** — Three completely different ways to use ESM-2 in three notebooks: frozen feature extractor (B1), fine-tuned (B2), zero-shot LM scoring (B3). When would you pick each?
4. **Real research connection** — Read the abstract of Meier et al. 2021 ("Language models enable zero-shot prediction of the effects of mutations on protein function"). Does what you just did match the paper's approach?

**Phase 3 B3 milestone:** A clean L×20 mutation heatmap, identification of conserved residues that match known catalytic/structural roles, and (if DMS data available) Spearman correlation ≥ 0.4 with experimental fitness.

# B3 — Zero-shot 突變效應預測（ESM-2）複習筆記

> 用一個從沒看過任何突變／功能資料的蛋白語言模型，光靠演化序列去預測突變的好壞。
> 本質上是親手複刻 **Meier et al. 2021**（"Language models enable zero-shot prediction of the effects of mutations on protein function"）的核心方法。

---

## 1. 這是什麼：Zero-shot 的意義

- **Zero-shot** = 完全沒用任何突變／功能資料去訓練。只拿預訓練好的 ESM-2 來問問題。
- ESM-2 唯一學過的任務是**填空**（masked language model，跟 BERT 一樣）：在數千萬條天然序列上，遮住一個位置去猜它是什麼胺基酸。
- 它從沒看過 fitness、活性、抗藥性 —— 它學到的是**「這個胺基酸出現在這個位置，在演化上有多典型／合理」**。
**為什麼這能預測 fitness？**（Reflection Q1 的核心）
> 天然序列是被天擇篩選過的。能定序到的蛋白，都是能摺疊、有功能、讓生物活下來的版本。
> 所以「演化上常見」暗藏了「功能上可行」。
> **演化本身就是一場跑了幾十億年的巨型突變篩選實驗，ESM-2 讀的是這場實驗的結果。**

---

## 2. 核心方法：Masked-marginal 打分

要問「這個位置該放什麼」，就把該位置**遮起來（mask）**，讓模型只根據其餘序列去預測 20 種胺基酸的機率分布。

- 一次只遮一個位置 → 建出 L 條各遮一格的序列 → 讓模型在預測某位置時能看到**最完整的脈絡**（除了那一格）。
- 這是 ESM variant-effect 論文的標準做法。
**分數定義：**
```
score[i, k] = log P(突變胺基酸_k | 脈絡) − log P(野生型胺基酸 | 脈絡)
```
是一個 **log 機率比值**：
- 分數為正 → 模型覺得突變比原本更「自然」→ 可能可容忍
- 分數很負 → 模型覺得突變很不可能 → 可能有害
- WT → WT 剛好 = 0
輸出是一個 **(L, 20) 矩陣** `S`。

---

## 3. 程式碼的兩個踩雷點

**`+1` 偏移**
`add_special_tokens=True` 會在序列前加 `<cls>`、後加 `<eos>`。
所以真正的第 `i` 個殘基，在 token 序列裡是 index `i+1`。所有遮罩、取 logits、取 WT id 的 `i+1` 都是為了對齊這個偏移。

**`aa_token_ids`**
ESM 詞彙表不只 20 個胺基酸（還有特殊符號、非標準殘基）。這個變數是「20 種標準胺基酸對應的 token id」，用來從整個 vocab 裡只切出那 20 欄。

---

## 4. Per-position tolerance（每個位置的容忍度）

對每個位置的 19 種非 WT 突變取平均 → 得到該位置的「演化容忍度」。
- 很負 = 高度保守 = 演化幾乎不允許改動 = 常是催化或結構錨點。
- 偏正 = 容忍度高 = 常是表面 / loop 上的位置。
**踩坑：`nanmean` vs `mean`**
> 用 `np.where(mask, S, np.nan)` 把 WT 那欄塞成 nan（只想平均剩下 19 個），
> 但接著用普通 `.mean(axis=1)` → **一整列只要有一個 nan，整列平均就是 nan** → 圖全空。
> 修法：改用 `np.nanmean(...)`，它會自動跳過 nan。

---

## 5. 關鍵案例：Ser70 與「序列合理性 ≠ 功能」

TEM-1 的催化 Ser70（position 70）:
- `tol[69]` = **−3.09**，比全序列平均 **−1.81** 更負 → ESM-2 確實認出它偏保守。
- **正確的論證方式**：不是「它是負的所以保守」，而是「它比背景更負，所以相對保守」。
**更深的檢查：**
```
Ser70 best substitution: +0.19   ← 竟然是正的！
Ser70 worst substitution: −5.46
```

**這個 +0.19 是整個 B3 最重要的洞察：**
> ESM-2 學的是「演化上什麼胺基酸會出現在這個脈絡」，**不是**「催化功能會不會保留」。
> best 的那個很可能是 **Thr**（跟 Ser 只差一個甲基、同有 −OH，在親緣蛋白這位置很常見）。
> 但 Ser70→Thr 幾乎肯定讓 β-lactamase 失活 —— 那個 −OH 的精確幾何是催化必需的。
> **ESM-2 看不到這一層，因為它從沒被教過「催化」，只被教過「序列長什麼樣」。**

⭐ **核心限制：ESM-2 打的是「演化合理性 / 序列自然度」，不是直接的「功能保留與否」。**
- 多數位置兩者一致 → 方法有用
- 催化殘基等特例會脫鉤 → 模型會低估某些保守性替換（Ser→Thr）的破壞力
---

## 6. 用 DMS 實驗資料驗證（section 7）

拿 ProteinGym 的 **BLAT_ECOLX_Stiffler_2015** —— Stiffler 等人在實驗室真的把 TEM-1 每個位點逐一突變、測 ampicillin 抗性的 ground truth，跟 `S` 矩陣做 Spearman 相關。

**資料 URL（舊的失效後更新）：**
```python
DMS_URL = 'https://huggingface.co/datasets/OATML-Markslab/ProteinGym_v0.1/resolve/main/ProteinGym_substitutions/BLAT_ECOLX_Stiffler_2015.csv'
```
欄位：`mutant`（如 `A23T`）、`DMS_score`（fitness）。

### ⭐ 最大的坑：座標系統不一致

第一次跑只配對到 **190 / 4996**（~4%，接近隨機撞對的比例）→ 紅燈。
不是模型爛，是**兩邊序列編號沒對齊**，配對整個歪掉。

**病因（生物學細節）：** TEM-1 有一段 **23 個殘基的訊號胜肽**。
ProteinGym 用含訊號胜肽的前驅蛋白編號，`WT_SEQ` 是成熟蛋白 → 整體平移 23 格。

**解法：讓資料自己招供 offset**（不靠翻文件）
- 每個 mutant 字串的第一個字母 = 該位置的野生型（`A23T` 宣告「23 號本來是 A」）。
- 用這點反推 DMS 的參考序列，再暴力試 −30~+30 的平移，數哪個平移讓最多位置的字母對上。
- 錯的 offset → 字母吻合率 ~5%；對的 offset → 接近 100%，出現尖銳高峰。
```python
dms_wt = {}
for m in dms_df[mut_col]:
    try: dms_wt[int(m[1:-1])] = m[0]
    except Exception: pass
positions = sorted(dms_wt)
best_off, best_hits = 0, -1
for off in range(-30, 31):
    hits = sum(1 for p in positions
               if 0 <= p-1+off < L and WT_SEQ[p-1+off] == dms_wt[p])
    if hits > best_hits: best_off, best_hits = off, hits
# 然後配對迴圈裡用 j = pos - 1 + best_off 取代 pos - 1
```

### 結果

```
Best offset = -23  (261/263 positions match)   ← 正好是訊號胜肽長度！
Matched mutations: 4958 / 4996                 ← 從 190 跳到 4958
Spearman correlation: 0.400                     ← milestone 達標
Pearson correlation:  0.431
```

**0.40 的意義：** 一個從沒看過任何突變實驗、不知道「抗藥性」是什麼的模型，
光靠讀天然序列的演化規律，就重現了四千多個突變效應約 40% 的排序趨勢。
（小 ESM-2 在 BLAT 上就是 ~0.4；650M 才能爬到 ~0.5。用小模型拿 0.40 完全合理。）

---

## 7. Reflection Questions 答案

**Q1 — 為什麼不用訓練就能做？**
ESM-2 只學過填空，預測的是「胺基酸在此位置的演化合理性」。因為天然序列是天擇篩選過的，「演化常見」暗藏「功能可行」，所以間接與 fitness 相關。演化 = 一場巨型突變篩選實驗。

**Q3 — 它會在哪裡失手？**（剩下沒對上的 60%）
核心：**ESM-2 評的是「單一位置的演化合理性」，超出這框架的效應都看不到。**
- **Epistasis（上位效應）**：masked-marginal 假設各位置獨立。但 A、B 單獨有害、A+B 一起卻互相補償（如正負電配對）—— 逐位置獨立打分結構上抓不到。
- **Allosteric sites（別構位點）**：不在活性中心、序列也不特別保守，但透過構型變化遠端調控活性。效應是動態／結構層面的，沒寫在序列保守性裡。
- **Gain-of-function（功能獲得）**：ESM-2 邏輯是「越像天然越好」→ 偏向偵測破壞型突變。但創造新功能的突變（如 β-lactamase 獲得分解新抗生素的能力）在演化上「不典型」（模型給負分），實驗 fitness 卻有利 —— 方向剛好看反。（= Ser70→Thr +0.19 的放大版）
**Q4 — B1 / B2 / B3 怎麼選？** 分水嶺是**手上有沒有標籤資料**：
| 方法 | 何時用 | 成本 / 天花板 |
|------|--------|--------------|
| **Zero-shot LM scoring (B3)** | 沒有標籤資料，純問演化直覺 | 最低 / 最低 (0.4–0.6) |
| **Frozen feature extractor (B1)** | 有少量標籤：抽 embedding 接小模型，ESM-2 不動 | 中 / 中 |
| **Fine-tuning (B2)** | 有較多標籤、任務差異大：連權重一起微調 | 最高 / 資料夠時最好 |

一句話：**沒資料→zero-shot；少資料→frozen features；多資料→fine-tune。**

**Q5 — 對照 Meier et al. 2021？**
**完全吻合。** 我做的 masked-marginal scoring（`log P(mut) − log P(wt)`）就是他們的主要打分策略；連拿 ProteinGym DMS 做驗證、用 Spearman 當指標，都跟論文的評估方式一致。這整個 B3 就是那篇論文的最小可執行版本。

---

## 8. Milestone 總結（三項全達成 ✓）

- ✅ L×20 mutation heatmap
- ✅ 保守殘基對上已知催化位點（Ser70 = −3.09，明顯低於背景 −1.81）
- ✅ DMS Spearman ≥ 0.4（= 0.400）
---

## 9. Debug 歷程（真實研究的縮影）

1. 圖畫不出來 → `nanmean` 的坑
2. 模型抓到 Ser70 了（−3.09 < 背景 −1.81）
3. 但 Ser70→Thr 為什麼是正分？→ **序列合理性 ≠ 功能**
4. 資料 URL 失效 → 換 HuggingFace ProteinGym_v0.1
5. 只配對 190/4996 的紅燈 → 座標沒對齊
6. 讓資料自己招出 **−23 offset**（= 訊號胜肽）
7. Spearman 0.40 達標
> 每一步都是真實研究會遇到的關卡 —— 這是一個縮小版但完整的計算生物學驗證研究。